# 04 — Model 2.1.1, typed flags as observations

**The model in math terms.** M2.1.1 is the mounted M1.2 stack (adopted Both chains, MIX2 routes, partition link) with one addition: typed misconception flags enter each home KC's chain update as extra likelihood factors. The observation view — one latent per KC, no disposition state, constant learn rate.

**State and transition.** Per KC, the belief $b_t = P(L_t = 1 \mid H_t)$, carried forward by the drift:

$$b^{\text{pre}}_t = b_{t-1} + (1 - b_{t-1})\,\tau$$

**Prediction** (before the turn reveals): the qc forecast pools the chains' act-probabilities through the MIX2 routes and the partition link $P = x(1-s_0) + (1-x)g_0$, exactly as in M1.2. Flags never enter prediction.

**Update** (after the turn reveals): the turn's data $D_t$ is the correctness symbol $o$ plus every flag symbol the annotation presents ($A_t$, the realized set — fired or quiet; NA is a structural non-observation). Symbols are conditionally independent given $L$, so the likelihood is a product:

$$P(D_t \mid L{=}l) = e_l(o) \times \prod_{j \in A_t} P(F_j = f_j \mid L{=}l)$$

with the correctness emissions $e_1(\text{correct}) = 1-s$, $e_0(\text{correct}) = g$ from the adopted chains, and the typed flag pair per flag $j$:

$$P(F_j{=}\text{fired} \mid L{=}1) = u_1 = 0.01 \text{ (pinned)}, \qquad P(F_j{=}\text{fired} \mid L{=}0) = u_0^{(j)} \text{ (fitted)}$$

The posterior is the two-hypothesis Bayes fraction, then the drift:

$$b_t = \frac{b^{\text{pre}}_t \, P(D_t \mid 1)}{b^{\text{pre}}_t \, P(D_t \mid 1) + (1 - b^{\text{pre}}_t)\, P(D_t \mid 0)}$$

A fire multiplies the odds by $u_1/u_0 \approx 0.02$–$0.05$ (the loudest wrong in the alphabet); a quiet by $(1-u_1)/(1-u_0) > 1$ (surviving a met trap outcredits a bare correct). An empty $A_t$ makes the product vanish and the model collapses to M1.2 exactly.

**Flag homing** (mechanism-based, per the report): conjunction → kc1, inverse → kc2, time-axis → kc2, denominator neglect → kc4, base-rate neglect → kc5.

**Flag-table fitting** (two-stage, matching the outer chain's precedent): with correctness-only responsibilities $w_t = 1 - b^{\text{pre}}_t$ on the home KC over training walks,

$$u_0^{(j)} = \frac{\sum_{t:\, j \in A_t} w_t \,\mathbb{1}[f_{j,t}{=}\text{fired}] + \kappa\, c_j}{\sum_{t:\, j \in A_t} w_t + \kappa}, \qquad \kappa = 5$$

shrunk toward the published written-format calibration $c_j$ (CPR uninstructed error rates: conjunction 0.79, inverse 0.65, time-axis 0.63, denominator 0.82, base-rate 0.67), clipped to $[0.1, 0.9]$. $u_1$ is never fitted. The estimator is uniform across all five flags and never conditioned on observed fire counts: conjunction (zero fires) runs through the same shrinkage, its quiet mass pulling the estimate below the calibration center, the pin surviving only as the $\kappa \to \infty$ limit (reconciliation ruling in the report).

**File layout.**
* The model: `scripts/model_2_1_1.py` (subclasses `Model_1_2_MIX2`)
* The ablation arms: `scripts/model_2_1_1_ablations.py` (run in section 3; not saved)
* The flag-blind comparison: **loaded from `cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv`** (written by notebook 03's `save_outer_chain_from_evaluator`) — M1.2 is *not* re-run in this notebook
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Harness `scripts/evaluator.py`, data `data/data_annotated.csv`, loader `scripts/data.py`
* Saved outputs: `cache/model_2_1_1/Model_2_1_1/` via `save_model_2_1_1_from_evaluator` (cell at the bottom)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [ ]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_1_1 import Model_2_1_1
from scripts.model_2_1_1_ablations import (Model_2_1_1_Joint, Model_2_1_1_No_Literature,
                                           Model_2_1_1_Joint_No_Literature, Model_2_1_1_Classic_BKT)
from scripts.model_1_1 import Model_1_1
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
MIX2_PREDS = 'cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv'   # saved by notebook 03's save helper; M1.2 is not re-run here

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
mix2 = pd.read_csv(MIX2_PREDS)
print(len(df), 'rows |', len(cache), 'cached folds |', len(mix2), 'stored flag-blind predictions')

312 rows | 26 cached folds | 312 stored flag-blind predictions


## 1. Run
M2.1.1 through the shared harness with the cached inner chains. The flag-blind comparison numbers come from the stored predictions, never re-fitted.

In [ ]:
ev = Evaluator(Model_2_1_1, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
preds = ev.predictions
print('done |', int(ev.metrics['n']), 'targets')

done | 312 targets


## 2. Results

### 2.1 Headline metrics against the stored flag-blind run
Metrics from the 312 pooled out-of-fold question predictions, same references as notebook 03 (qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.653).

In [ ]:
pd.DataFrame([dict(model='M2.1.1', **{k: round(float(v),4) for k,v in ev.metrics.items()}),
              dict(model='M1.2 (stored)', **{k: round(float(v),4) for k,v in _metrics(mix2.y_true, mix2.p_pred).items()})]
             ).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy','f1','n']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy,f1,n
model,,,,,,,
M2.1.1,0.6907,0.5908,0.6073,0.6002,0.6955,0.7948,312.0
M1.2 (stored),0.6741,0.5747,0.6018,0.6070,0.6859,0.7860,312.0


### 2.2 Where the gain lives
The attribution split: ever-fired participants against never-fired. Note the label carefully — quiets are typed evidence too and update every participant's chains, so this split isolates who ever fired, not who was affected by the flag channel.

In [ ]:
FLAGGED = ['P01','P02','P03','P06','P11','P20','P23','P24']
def grp(p, pids):
    sub = p[p.participant_id.isin(pids)]
    return round(float(_metrics(sub.y_true, sub.p_pred)['auc']), 3)
others = sorted(set(df.participant_id) - set(FLAGGED))
pd.DataFrame([
    dict(group='flag-bearers', m2_1_1=grp(preds, FLAGGED), m1_2=grp(mix2, FLAGGED)),
    dict(group='zero-fire',    m2_1_1=grp(preds, others),  m1_2=grp(mix2, others)),
]).set_index('group')

,m2_1_1,m1_2
group,,
flag-bearers,0.663,0.636
zero-fire,0.659,0.661


#### Row-level gains and losses
`good` is movement toward the truth: the prediction change on corrects, its negative on wrongs. The tables show the rows the flag channel helped most and hurt most, then the per-participant and per-question AUC deltas.

In [ ]:
j = preds.merge(mix2, on=['participant_id','question_number'], suffixes=('_2','_1'))
j['delta'] = j.p_pred_2 - j.p_pred_1
j['good'] = np.where(j.y_true_2 == 1, j.delta, -j.delta)
j['flagged'] = j.participant_id.isin(FLAGGED)
print(f'rows helped (good > 0.01): {int((j.good > 0.01).sum())} | rows hurt: {int((j.good < -0.01).sum())}')
print('biggest helps:')
display(j.nlargest(6, 'good')[['participant_id','question_number','p_pred_1','p_pred_2','y_true_2']].round(3))
print('biggest hurts:')
display(j.nsmallest(6, 'good')[['participant_id','question_number','p_pred_1','p_pred_2','y_true_2']].round(3))

rows helped (good > 0.01): 85 | rows hurt: 41
biggest helps:


,participant_id,question_number,p_pred_1,p_pred_2,y_true_2
232,P20,5,0.706,0.513,0
280,P24,5,0.782,0.605,0
28,P03,5,0.793,0.619,0
7,P01,8,0.462,0.317,0
281,P24,6,0.607,0.475,0
267,P23,4,0.460,0.348,0


biggest hurts:


,participant_id,question_number,p_pred_1,p_pred_2,y_true_2
68,P06,9,0.743,0.482,1
16,P02,5,0.771,0.576,1
124,P11,5,0.774,0.586,1
6,P01,7,0.428,0.323,1
287,P24,12,0.329,0.245,1
285,P24,10,0.395,0.317,1


In [ ]:
rows = []
for pid, g in j.groupby('participant_id'):
    a1 = _metrics(g.y_true_1, g.p_pred_1)['auc']; a2 = _metrics(g.y_true_2, g.p_pred_2)['auc']
    if a1 == a1 and a2 == a2:
        rows.append(dict(participant=pid, m1_2=round(a1,3), m2_1_1=round(a2,3),
                         delta=round(a2-a1,3), flagged=pid in FLAGGED))
pd.DataFrame(rows).sort_values('delta').set_index('participant')

,m1_2,m2_1_1,delta,flagged
participant,,,,
P11,0.531,0.438,-0.094,True
P16,0.500,0.450,-0.050,False
P12,0.667,0.630,-0.037,False
P02,0.750,0.719,-0.031,True
P06,0.543,0.514,-0.029,True
P22,0.818,0.818,0.000,False
P21,0.909,0.909,0.000,False
P18,0.909,0.909,0.000,False
P15,0.750,0.750,0.000,False


In [ ]:
rows = []
for q, g in j.groupby('question_number'):
    a1 = _metrics(g.y_true_1, g.p_pred_1)['auc']; a2 = _metrics(g.y_true_2, g.p_pred_2)['auc']
    if a1 == a1 and a2 == a2:
        rows.append(dict(question=q, m1_2=round(a1,3), m2_1_1=round(a2,3), delta=round(a2-a1,3)))
pd.DataFrame(rows).sort_values('delta').set_index('question')

,m1_2,m2_1_1,delta
question,,,
11,0.519,0.504,-0.015
1,0.034,0.023,-0.011
6,0.556,0.549,-0.007
3,0.120,0.120,0.000
7,0.458,0.458,0.000
10,0.799,0.799,0.000
12,0.825,0.825,0.000
9,0.739,0.745,0.007
2,0.210,0.219,0.010


### 2.3 Fitted flag tables and bridge anchors
Fold-mean fitted fire rates beside their written-format calibration centers, and the bridge anchors against the mechanism censuses.

In [ ]:
u0 = pd.DataFrame([m.u0 for m in ev.fold_models.values()]).mean().round(3)
anchors = pd.Series(dict(s0=np.mean([m.s0 for m in ev.fold_models.values()]),
                         g0=np.mean([m.g0 for m in ev.fold_models.values()]))).round(3)
print('fitted u0 (fold means) vs written-format calibration:')
display(pd.DataFrame(dict(fitted=u0, calibration=pd.Series({'conjunction':0.79,'inverse':0.65,'time_axis':0.63,'denominator_neglect':0.82,'base_rate_neglect':0.67}))))
print('bridge anchors (censuses 0.077 / 0.061):')
display(anchors)

fitted u0 (fold means) vs written-format calibration:


,fitted,calibration
conjunction,0.644,0.79
inverse,0.468,0.65
time_axis,0.605,0.63
denominator_neglect,0.282,0.82
base_rate_neglect,0.211,0.67


bridge anchors (censuses 0.077 / 0.061):


s0    0.080
g0    0.092
dtype: float64

### 2.4 Confusion matrices
Threshold 0.5, correct as the positive class, M2.1.1 beside the stored flag-blind M1.2.

**Observations.**
* Both matrices share the same shape of error: the dominant failure cell is wrongs predicted correct (false positives) — at this threshold most wrong answers arrive unannounced under either model.
* At the 0.5 threshold M2.1.1 does **not** shrink the blindside set: the net movement is FP 78 to 79 and FN 20 to 16. The threshold-level gain comes from recovering previously missed corrects (the model becomes slightly more willing to predict correct), not from catching more wrongs — wrong-recall dips from 34/112 to 33/112. The improvement the flags deliver is ranking-shaped (AUC, AUPRC-wrong, log-loss all improve); the arbitrary 0.5 line barely feels it.
* The residual false positives are dominated by never-fired construal and misread participants (P16, P26, P25, P15, P10) — the blindside mass the flag channel cannot see by construction, and the measured motivation for the parked misinterpretation channel.


In [ ]:
def cmat(p):
    yhat = (p.p_pred >= 0.5).astype(int)
    cm = pd.crosstab(p.y_true.map({1:'actual correct', 0:'actual wrong'}),
                     yhat.map({1:'predicted correct', 0:'predicted wrong'}))
    return cm.reindex(index=['actual correct','actual wrong'],
                      columns=['predicted correct','predicted wrong'], fill_value=0)

cm2, cm1 = cmat(preds), cmat(mix2)
print('M2.1.1:'); display(cm2)
print('M1.2 (stored):'); display(cm1)
fp2 = cm2.loc['actual wrong','predicted correct']; fp1 = cm1.loc['actual wrong','predicted correct']
fn2 = cm2.loc['actual correct','predicted wrong']; fn1 = cm1.loc['actual correct','predicted wrong']
print(f'false positives {fp1} -> {fp2} | false negatives {fn1} -> {fn2}')

M2.1.1:


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,184,16
actual wrong,79,33


M1.2 (stored):


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,180,20
actual wrong,78,34


false positives 78 -> 79 | false negatives 20 -> 16


## 3. Ablations

Four arms probing the canonical model's assumptions, each with its math. None is saved; the canonical model's save cell at the bottom is the only persistence.

**3a. Joint emission, the faithful-Feng form (independence removed).** Per home KC per turn the outcome is one symbol from a mutually exclusive alphabet, correct, wrong-with-flag-$j$-fired, wrong-all-quiet, with the correct-and-fired cell a structural zero. Writing $q_l^{(j)} = P(\text{fired}_j \mid \text{wrong}, L{=}l)$:

$$P(\text{c} \mid l) = e_l(\text{c}), \qquad P(\text{w}, f_j \mid l) = e_l(\text{w})\, q_l^{(j)}, \qquad P(\text{w}, \varnothing \mid l) = e_l(\text{w}) \prod_j (1 - q_l^{(j)})$$

$q_1$ pinned at 0.01, $q_0$ fitted with a light neutral-center shrink ($\kappa_J = 1$ toward 0.5). The wrong's evidence is counted once, inside $e_l(\text{w})$; the flag adds only which-kind-of-wrong. Flag-only turns (cell NA) use the implied marginal $r_l = q_l\, e_l(\text{w})$.

**3b. No-literature (calibration deleted).** The canonical factorized form with the prior removed entirely: $u_0^{(j)} = $ the responsibility-weighted fire rate alone, $\kappa = 0$, clip floor lowered to 0.01 so no knowledge enters through the bounds either. Channels with no training fires collapse to the floor and go inert.

**3c. Joint, no-literature.** Both deletions at once: the joint form with $q_0$ also $\kappa$-free at the 0.01 floor. The harshest cell of the square; its thin conditionals (denominators are wrongs only) saturate without shrink.

**3d. Classic BKT plus flags (on Model_1_1).** The user's Model_1_1 chassis unchanged: chains trained on pseudo-turns (the qc label broadcast to every designed KC), prediction the compensatory mean over the designed set, updates the qc label broadcast, the per-cell annotation unused by design. The flags are the sole cell-layer information admitted: each presented flag multiplies its typed factor (the registered $\kappa{=}5$ tables, $u_1$ pinned) into its home chain's update during the walk,

$$b_k \leftarrow \text{Bayes}\big(b_k;\ e_l(y) \times \prod_{j:\, \text{home}(j)=k} P(F_j = f_j \mid L{=}l)\big) \ \text{then drift}$$

with a flag whose home KC is outside the designed set updating that chain alone (flag-only Bayes plus drift). Against flag-blind Model_1_1 this asks whether the typed channel adds signal to a chassis that never sees cells.

In [ ]:
kwa = dict(n_restarts=3, chain_cache=cache)
arms = {}
for cls in (Model_2_1_1_Joint, Model_2_1_1_No_Literature, Model_2_1_1_Joint_No_Literature):
    arms[cls.__name__] = Evaluator(cls, df, model_kwargs=kwa).run()
    print(cls.__name__, 'done')
for cls in (Model_1_1, Model_2_1_1_Classic_BKT):
    arms[cls.__name__] = Evaluator(cls, df, model_kwargs=dict(n_restarts=3)).run()
    print(cls.__name__, 'done')

Model_2_1_1_Joint done
Model_2_1_1_No_Literature done
Model_2_1_1_Joint_No_Literature done
Model_1_1 done
Model_2_1_1_Classic_BKT done


In [ ]:
rows = [dict(model='M2.1.1 canonical (factorized, calibrated)',
             **{k: round(float(v),4) for k,v in ev.metrics.items()})]
labels = {'Model_2_1_1_Joint': 'Joint, calibrated (faithful-Feng)',
          'Model_2_1_1_No_Literature': 'Factorized, no-literature',
          'Model_2_1_1_Joint_No_Literature': 'Joint, no-literature',
          'Model_2_1_1_Classic_BKT': 'Classic BKT + flags (Model_1_1 chassis)',
          'Model_1_1': 'Model_1_1 flag-blind (classic reference)'}
for name, e in arms.items():
    rows.append(dict(model=labels[name], **{k: round(float(v),4) for k,v in e.metrics.items()}))
rows.append(dict(model='M1.2 flag-blind (mounted reference)',
                 **{k: round(float(v),4) for k,v in _metrics(mix2.y_true, mix2.p_pred).items()}))
pd.DataFrame(rows).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy
model,,,,,
"M2.1.1 canonical (factorized, calibrated)",0.6907,0.5908,0.6073,0.6002,0.6955
"Joint, calibrated (faithful-Feng)",0.6870,0.5834,0.6227,0.6028,0.7051
"Factorized, no-literature",0.6867,0.5858,0.6138,0.6031,0.6987
"Joint, no-literature",0.6835,0.5758,0.6227,0.6053,0.7051
Model_1_1 flag-blind (classic reference),0.5974,0.5163,0.5971,0.6283,0.7051
Classic BKT + flags (Model_1_1 chassis),0.6103,0.5321,0.6184,0.6223,0.7147
M1.2 flag-blind (mounted reference),0.6741,0.5747,0.6018,0.6070,0.6859


**Observations.**
* **The emission-form-by-priors square is additive and small.** Each deletion (independence, literature) costs ~0.004 AUC alone, ~0.0075 together, and the primary contrast survives the harshest cell (joint no-lit ~0.683 vs M1.2 ~0.674): the headline claim rests on the flags, not on the priors and not on the independence assumption.
* **The joint form is the correct theory, the factorized form the adopted estimator.** The joint emission describes the instrument faithfully (fires entail the home cell's failure; the structural zero encodes it) but its conditionals divide by wrongs only, thinner denominators that saturate without shrink (unshrunk time-axis $q_0 \to 0.99$, flipping wrong-without-fire into evidence of mastery). The factorized double-count is directionally helpful in a persistence-dominated cohort. Misspecified-but-robust beats faithful-but-fragile at this n; at larger n the joint form is the right canonical.
* **The literature buys the thin channels, not the performance.** Without calibration, conjunction and inverse collapse to the floor and go inert while the pooled result barely moves; the verbal-suppression finding strengthens (unshrunk BRN ~0.084 vs written 0.67).
* **The mechanism travels.** On the classic chassis that never sees cells, the flags still add ~+0.013 AUC over flag-blind Model_1_1, with the gain concentrated in flag-bearers: the typed channel is not an artifact of the mounted stack or the cell annotation.
* **Adjudication: the canonical M2.1.1 stays the factorized, calibrated form** — pre-registered, with the best proper scores in this run (the between-arm differences sit within noise pending intervals), and the chassis every downstream model mounts on — with the joint form co-reported as the fidelity variant.

## 4. Conclusion

* **The typed flags add predictive signal at the mounted qc layer.** M2.1.1 beats the flag-blind M1.2 on every headline metric (AUC ~0.691 vs ~0.674, AUPRC-wrong ~0.591 vs ~0.575, log-loss ~0.600 vs ~0.607) on the same 312 targets with identical inner chains.
* **The gains have a mechanism, post-fire propagation.** The largest improvements are the turns immediately after a fire: the Q4 time-axis fires crash the kc2 chain and the Q5 predictions inherit the crash, correctly, for the students who stayed wrong (P20, P24, P03); P01's denominator fires do the same for her later kc4 items. A fire acts as a one-turn-ahead warning about its home KC.
* **The losses are the same mechanism backfiring on recoverers, and they concentrate on the trap-free probes.** Students who fired and then survived the next turn are over-penalized (P02 and P11 at Q5, P06 at Q9, P01 at Q7). Q5 and Q9 are the designed trap-free probes, and P06's Q9 is the defining case: machinery intact, habit live, answer right, prediction dragged to 0.48. The observation view has no cell for mastered-but-trapped, so the fire's damage lands on mastery and propagates onto turns the trap cannot touch. This is the registered M2.1.1 vs M2.1.2 fork visible in the residuals, and the empirical case for the state view.
* **The ever-fired group splits accordingly.** Persisters gain and recoverers lose (approximate per-participant AUC deltas in section 2.2's table; exact values are environment-dependent — they move with the calibration constants and the chain cache, so read them from this run's outputs rather than from prose). Several never-fired participants show exactly zero AUC delta; note this reflects ranking insensitivity to small monotone shifts, not absence of effect — their quiets still update the chains. The net is positive because persistence outweighs recovery in this cohort.
* **Verbal-format fire suppression, quantified.** The fitted unmastered fire rates sit far below their written-format calibrations (base-rate ~0.21 vs 0.67; denominator ~0.31 vs 0.82): students explaining aloud fire the canonical fallacies at a third to half their written rates, the datum the report pre-registered as of interest.
* **The bridge stays honest.** Anchors land at s0 ~0.080 (census 0.077) and g0 ~0.091 (census 0.061), the tightest census match yet: the flag evidence sharpens the chains without stealing the bridge's meaning.
* **Caveats.** The pooled effect is modest (+0.017 AUC) and participant-clustered bootstrap intervals are pending; the gains are ranking-shaped (few threshold crossings, section 2.4); the residual blindsides belong to the zero-fire construal and misread participants (the parked misinterpretation channel's territory); and this is the mounted qc secondary, with the registered per-KC primary contrast still to run.


## 5. Save
Persist the adopted model only, the ablation arms are not saved. Per-fold bridge and flag tables, pooled predictions, metrics, index: per-fold bridge and flag tables, pooled predictions, metrics, and the index (pins, kappa, calibration, homing).

In [ ]:
import os
import json
from scripts.model_2_1_1 import save_model_2_1_1_from_evaluator

out_dir = 'cache/model_2_1_1'
save_model_2_1_1_from_evaluator(ev, out_dir)

'cache/model_2_1_1/Model_2_1_1'